# ⑤ 평가 — 네 가지로 나눠 채점
담당: 평가 · 코드: `rag/evaluation/evaluation.py`

04 생성의 `artifacts/answers.jsonl` 하나를 네 가지로 채점해, 나쁜 답이 검색에서 왔는지 · 생성에서 왔는지 가릅니다.
RAGAS 심판은 `gpt-5-mini` (OpenAI API, 유료)입니다. 각 표의 마지막 행(`평균`)이 요약입니다.
04 를 Langfuse 키와 함께 실행했다면 질문별 점수가 그 질문의 Langfuse 추적에도 붙습니다 (지연 시간 · 토큰 · 비용은 추적에 이미 있어 점수로 보내지 않음).

| 평가 | 판단하는 것 | 저장 파일 |
|---|---|---|
| Retrieval | 검색한 청크가 정답에 필요한 내용을 담았나 | `artifacts/retrieval.csv` |
| Generation | 답이 질문에 맞고 핵심 사실을 담았나 | `artifacts/generation.csv` |
| Grounding | 답이 검색한 청크 안에서만 나왔나 (환각) | `artifacts/grounding.csv` |
| End-to-End | 최종 답이 정답과 맞나 · 시간 · 토큰 · 비용 | `artifacts/end_to_end.csv` |

In [ ]:
from rag.config import ARTIFACTS, load_config
from rag.evaluation.evaluation import (evaluate_end_to_end, evaluate_generation, evaluate_grounding,
                                       evaluate_retrieval)

cfg = load_config()
ANSWERS = ARTIFACTS / "answers.jsonl"

## Retrieval
- `context_precision`: 검색한 청크 중 정답에 쓸모 있는 청크가 위에 있나 (RAGAS)
- `context_recall`: 정답의 내용이 검색한 청크에 다 있나 (RAGAS)
- `key_facts_in_context`: 핵심 사실 중 검색한 청크에 들어 있는 비율

In [ ]:
retrieval = evaluate_retrieval(ANSWERS, cfg)
retrieval.to_csv(ARTIFACTS / "retrieval.csv", index=False, encoding="utf-8")
retrieval

## Generation
- `answer_relevancy`: 답이 질문에 맞는 답인가 (RAGAS, KURE-v1 임베딩)
- `key_facts_in_answer`: 핵심 사실 중 답에 들어 있는 비율

In [ ]:
generation = evaluate_generation(ANSWERS, cfg)
generation.to_csv(ARTIFACTS / "generation.csv", index=False, encoding="utf-8")
generation

## Grounding
- `faithfulness`: 답의 주장 중 검색한 청크로 뒷받침되는 비율 (RAGAS) — 낮으면 환각

In [ ]:
grounding = evaluate_grounding(ANSWERS, cfg)
grounding.to_csv(ARTIFACTS / "grounding.csv", index=False, encoding="utf-8")
grounding

## End-to-End
- `factual_correctness`: 답과 정답의 사실 일치 F1 (RAGAS)
- `latency_s`: 질문 하나의 검색 + 생성 시간(초)
- `input_tokens` · `output_tokens` · `reasoning_tokens`: 생성 토큰 (reasoning 은 output 에 포함)
- `cost_usd`: `config.yaml` 의 `llm.usd_per_1m_tokens` 로 계산한 예상 비용

In [ ]:
end_to_end = evaluate_end_to_end(ANSWERS, cfg)
end_to_end.to_csv(ARTIFACTS / "end_to_end.csv", index=False, encoding="utf-8")
end_to_end